# EuroSAT Review 2 — Colab GPU
Extract the delivered ZIP to Drive. Select Runtime → Change runtime type → GPU. The full protocol requires 1,073 unique trainings and approximately 45–50 GB for checkpoints. Run stages in chunks; completed jobs are reused. Read README.md first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
PACKAGE_DIR = Path('/content/drive/MyDrive/eurosat-review2')  # EDIT
DATA_DIR = Path('/content/EuroSAT_RGB')  # EDIT: ten class folders directly here
OUTPUT_DIR = Path('/content/drive/MyDrive/eurosat-review2-results')
os.chdir(PACKAGE_DIR)
assert Path('review_workflow.py').exists()

In [ ]:
%pip install -r requirements-bootstrap.txt

Restart the runtime if installation requests it, then rerun the setup/import cells. Keep Colab’s CUDA-enabled PyTorch. Do not install the bundled CPU test snapshot on Colab.

In [ ]:
import torch, torchvision
print(torch.__version__, torchvision.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'environment_capture.py', '--output', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'test_phase1', 'test_review2'], check=True)

In [ ]:
def stage(name, limit=None):
    command = [sys.executable, 'review_workflow.py', name, '--data-dir', str(DATA_DIR), '--output', str(OUTPUT_DIR)]
    if limit is not None: command += ['--limit', str(limit)]
    subprocess.run(command, check=True)
stage('prepare')
stage('plan')

## gn-search
Rerun this cell until this stage is complete, then proceed. For interrupted jobs, see `--restart-incomplete` in README.

In [ ]:
stage('gn-search', limit=5)

## select-gn
This stage refuses incomplete prerequisites.

In [ ]:
stage('select-gn')

## loss-search
Rerun this cell until this stage is complete, then proceed. For interrupted jobs, see `--restart-incomplete` in README.

In [ ]:
stage('loss-search', limit=5)

## select-losses
This stage refuses incomplete prerequisites.

In [ ]:
stage('select-losses')

## final-train
Rerun this cell until this stage is complete, then proceed. For interrupted jobs, see `--restart-incomplete` in README.

In [ ]:
stage('final-train', limit=5)

## seal
This stage refuses incomplete prerequisites.

In [ ]:
stage('seal')

## evaluate
Rerun this cell until this stage is complete, then proceed. For interrupted jobs, see `--restart-incomplete` in README.

In [ ]:
stage('evaluate', limit=5)

## report
This stage refuses incomplete prerequisites.

In [ ]:
stage('report')

After completion, use package_models.py to archive real checkpoints with SHA-256 checksums. Complete the release checks and Git tags as described in README.md. The historical environment and unavailable original checkpoints cannot be reconstructed from metrics alone.